In [ ]:
import pandas as pd

import os

base_path = os.path.dirname(os.getcwd())  # go one level up
data_path = os.path.join(base_path, "data")

X = pd.read_csv(os.path.join(data_path, "X_features.csv"))
y = pd.read_csv(os.path.join(data_path, "y_labels.csv")).squeeze()

In [ ]:
print(X.shape)
print(y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
import os

os.makedirs("../models", exist_ok=True)

In [ ]:
import joblib

joblib.dump(scaler, "../models/scaler.pkl")

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

In [ ]:
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, y_pred)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()
# save plot to results directory as pdf
results_dir = os.path.join(base_path, "results")
os.makedirs(results_dir, exist_ok=True)
plt.savefig(os.path.join(results_dir, "roc_curve.pdf"), dpi=300)

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, _ = precision_recall_curve(y_test, y_prob)

plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

results_dir = os.path.join(base_path, "results")
os.makedirs(results_dir, exist_ok=True)
plt.savefig(os.path.join(results_dir, "precision_recall_curve.pdf"), dpi=300)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)  # NOTE: no scaling needed

In [ ]:
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, y_pred_rf)

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, y_pred_rf)

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

fpr, tpr, _ = roc_curve(y_test, y_prob_rf)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")  
plt.legend()
plt.show()
# save plot to results directory as pdf
results_dir = os.path.join(base_path, "results")
os.makedirs(results_dir, exist_ok=True)
plt.savefig(os.path.join(results_dir, "roc_curve_rf.pdf"), dpi=300)

In [ ]:
#compare models for logistic and random forest for accuracy and AUC and make table 
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))
print(classification_report(y_test, y_pred_rf))

The model performs reasonably on non-binding interactions but fails to identify binding interactions effectively, as seen by the very low recall for class 1. This suggests class imbalance and insufficient feature representation. Improving recall for binding class is critical for drug discovery applications.

Extract feature importance (Random Forest)

In [ ]:
import pandas as pd

importances = rf_model.feature_importances_
feature_names = X.columns

feat_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

In [ ]:
feat_df.head(20)

In [ ]:
import matplotlib.pyplot as plt

top = feat_df.head(10)

plt.figure(figsize=(8,5))
plt.barh(top["Feature"], top["Importance"])
plt.gca().invert_yaxis()
plt.title("Top 10 Important Features")
plt.xlabel("Importance")
plt.show()

In [ ]:
feat_df["Type"] = feat_df["Feature"].apply(
    lambda x: "Protein" if x in list("ACDEFGHIKLMNPQRSTVWY") + ["Seq_Length"] else "Ligand"
)

feat_df.groupby("Type")["Importance"].sum()

Feature importance analysis shows that protein-derived features dominate the model, contributing ~96% of the total importance. This suggests that the current ligand representation is insufficient to capture chemical variability, leading the model to rely primarily on protein sequence features. In reality, protein?ligand binding depends on both molecular structure and protein context, indicating a limitation in the current feature engineering approach.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

# Create generator (once, outside function ideally)
morgan_gen = GetMorganGenerator(radius=2, fpSize=1024)

def get_ligand_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    
    if mol is None:
        return None

    # Descriptors
    features = [
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumRotatableBonds(mol),
        Descriptors.RingCount(mol),
        Descriptors.HeavyAtomCount(mol)
    ]

    # NEW fingerprint method
    fp = morgan_gen.GetFingerprint(mol)
    fp_list = list(fp)

    return features + fp_list

In [ ]:
import pandas as pd

df = pd.read_csv("../data/cleaned_dataset_valid.csv")

ligand_features = df["SMILES"].apply(get_ligand_features)

# Remove invalid rows
valid_idx = ligand_features.notnull()
df = df[valid_idx].reset_index(drop=True)
ligand_features = ligand_features[valid_idx].reset_index(drop=True)

In [ ]:
ligand_df = pd.DataFrame(ligand_features.tolist())

In [ ]:
ligand_df.to_csv("../data/ligand_features_updated.csv", index=False)

In [ ]:
protein_df = pd.read_csv("../data/protein_features.csv").reset_index(drop=True)
ligand_df = pd.read_csv("../data/ligand_features_updated.csv").reset_index(drop=True)
df = pd.read_csv("../data/cleaned_dataset_valid.csv").reset_index(drop=True)

In [ ]:
min_len = min(len(protein_df), len(ligand_df), len(df))

protein_df = protein_df.iloc[:min_len]
ligand_df = ligand_df.iloc[:min_len]
df = df.iloc[:min_len]

In [ ]:
X = pd.concat([protein_df, ligand_df], axis=1)
y = df["Label"]

In [ ]:
X.to_csv("../data/X_features_updated.csv", index=False)
y.to_csv("../data/y_labels_updated.csv", index=False)

RE-RUN ML PIPELINE

In [ ]:
X = pd.read_csv("../data/X_features_updated.csv")
y = pd.read_csv("../data/y_labels_updated.csv").squeeze()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

log_model.fit(X_train_scaled, y_train)

y_pred_log = log_model.predict(X_test_scaled)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report

print("Logistic:\n", classification_report(y_test, y_pred_log))
print("Random Forest:\n", classification_report(y_test, y_pred_rf))

After enhancing ligand representation using higher-dimensional Morgan fingerprints (1024 bits) and additional molecular descriptors, the model showed a significant improvement in detecting binding interactions. The recall for the binding class increased from ~0.15 to ~0.57, indicating that richer chemical features are critical for capturing protein?ligand interactions. This highlights the importance of proper feature engineering in cheminformatics-driven machine learning models.

In [ ]:
y_prob = log_model.predict_proba(X_test_scaled)[:,1]

# Try threshold 0.4 instead of 0.5
y_pred_custom = (y_prob > 0.4).astype(int)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=100)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print("Custom Threshold (0.4):\n")
print(classification_report(y_test, y_pred_custom))

Lowering the classification threshold from 0.5 to 0.4 significantly increased recall for binding interactions (from ~0.57 to ~0.88), demonstrating the model?s ability to capture most potential binders. However, this came at the cost of a drastic drop in specificity, with non-binding recall falling to ~0.09. This highlights the trade-off between sensitivity and precision, which is critical in drug discovery workflows where early-stage screening prioritizes high recall, while later stages require higher precision.

In [ ]:
y_pred_custom = (y_prob > 0.45).astype(int)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print("Custom Threshold (0.45):\n")
print(classification_report(y_test, y_pred_custom))

A threshold of 0.45 provided the best balance between sensitivity and specificity. The model achieved a recall of ~0.74 for binding interactions while maintaining some ability to detect non-binding cases. This balance is particularly suitable for early-stage drug discovery, where identifying potential binders is important, but excessive false positives should also be controlled.

In [ ]:
#PCA is used to reduce dimensionality and remove noise from high-dimensional fingerprint features.
from sklearn.decomposition import PCA

pca = PCA(n_components=100)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_pca = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)

rf_pca.fit(X_train_pca, y_train)

In [ ]:
from sklearn.metrics import classification_report

y_pred_rf_pca = rf_pca.predict(X_test_pca)

print("Random Forest with PCA:\n")
print(classification_report(y_test, y_pred_rf_pca))

| Model           | Recall (1)  | Precision (1) | F1   |
| --------------- | ----------- | ------------- | ---- |
| Logistic (0.5)  | 0.57        | 0.47          | 0.51 |
| Logistic (0.45) | **0.74 ??** | 0.40          | 0.52 |
| RF              | 0.17        | 0.46          | 0.25 |
| RF + PCA        | 0.14        | 0.40          | 0.20 |

Your best model is:
Logistic Regression
Threshold = 0.45


Random Forest performance remained poor even after applying PCA, with very low recall for binding interactions (~0.14). This indicates that dimensionality reduction alone is insufficient to improve model performance. In contrast, Logistic Regression performed significantly better, suggesting that the relationship between features and binding interactions is better captured by a linear model in this dataset.

In [ ]:
import os, joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(log_model, "../models/final_model.pkl")
joblib.dump(scaler, "../models/scaler.pkl")

In [ ]:
import joblib

feature_names = X.columns.tolist()
joblib.dump(feature_names, "../models/feature_names.pkl")